# Kinetic parameters

We use features of the data to obtain estimates of the following
- Bias of progenitor population to differentiation
- Average cell division rate of dividing cells
- Average cell delamination rate for delaminating cells
- Average shed rate for suprabasal cells
- Fraction of proliferating cells


We note the biases and assumptions of these calculations in the Supplemental Note.


In [9]:
# Setting up experimental data 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("../../analysis/")
sys.path.append("../../sim/")
import clones

plt.rcParams['figure.figsize'] = (8, 5)


import pickle

def save_object(obj, filename):
    with open(filename, 'wb') as outp:  # Overwrites any existing file.
        pickle.dump(obj, outp, pickle.HIGHEST_PROTOCOL)

def load_object(filename):
    with open(filename, 'rb') as outp:  # Overwrites any existing file.
        return pickle.load(outp)
    
def Sclones(livesize):
    nclones = livesize.shape[0]
    max_tot = np.max(np.sum(livesize,2),1)
    
    Slike = (max_tot >= 3)
    return livesize[Slike,:,:]
    
def Tclones(livesize):
    nclones = livesize.shape[0]
    max_tot = np.max(np.sum(livesize,2),1)
    
    Tlike = (max_tot == 2)
    return livesize[Tlike,:,:]

def Dclones(livesize):
    nclones = livesize.shape[0]
    max_tot = np.max(np.sum(livesize,2),1)
    
    Dlike = (max_tot == 1)
    return livesize[Dlike,:,:]


# Functions for curve-fitting and extraction of parameters

import statsmodels.api as sm
from scipy.optimize import curve_fit


def wls_fit(x, y, se):
    # Prepare the data for statsmodels (add constant term for intercept)
    X = sm.add_constant(x)

    # Perform the weighted fit
    model = sm.WLS(y, X, weights=1/se**2)
    results = model.fit()

    # Extract fitted parameters
    m, c = results.params[1], results.params[0]
    x = np.linspace(0, x[-1], 100)
    dm = results.bse[1] 
    dc = results.bse[0]

    return m, c, dm, dc

def Aexp_fn(t, μ, A):
    return A*np.exp(μ*t)

def get_mu(clone_anal):
    clone_anal = clone_anal.Dlike()
    if clone_anal.nsur[1] > 0:
        se = clone_anal.se_lcd
        se[se <= 0] = 1/np.sqrt(clone_anal.nclones)/2
        popt, pcov = curve_fit(D_exp_fn, clone_anal.tvals, clone_anal.lcd, sigma = se, 
                    absolute_sigma = True, bounds = (0,5))
        
        μ_fit = popt[0]
        μ_err = np.sqrt(np.diag(pcov))[0]
    
        return μ_fit, μ_err
    else:
        return np.nan, np.nan
    
def D_exp_fn(t, μ):
    return np.exp(-μ*(t - 7.0))*(t >= 7)  + (t < 7)


def bias_to_division(clone_anal, nboots = 1000, tlower = 3):
    ''' 
        Estimation of bias to division for proliferative cells in data
    '''

    Δλ, c, dm, dc = wls_fit(clone_anal.tvals[tlower:], np.log(clone_anal.lcd[tlower:]), clone_anal.se_lcd[tlower:])
    '''
    Δλ, se_Δλ  = curve_fit(Aexp_fn, clone_anal.tvals[tlower:], clone_anal.lcd[tlower:], sigma = clone_anal.se_lcd[tlower:], 
                        absolute_sigma = True, bounds = ((-1,0), (1, 5)))
    '''

    lcd_boots = clone_anal.boot('lcd', nboots, calc = None)
    boot_Δλ = np.zeros(nboots)
    for b in range(nboots):
        boot_Δλ[b], c, dm, dc = wls_fit(clone_anal.tvals[tlower:], np.log(lcd_boots[b,tlower:]), clone_anal.se_lcd[tlower:])

    return Δλ, np.std(boot_Δλ)




exp_folder = '../../experiment/homeo/'

colm = {'M30': 'r', 'M1': 'b', 'M14': 'g', 'M3': 'tab:orange', 'M1bis': 'tab:purple',
        'F3':  'r', 'F4': 'b', 'F11': 'g', 'F9': 'tab:orange', 'F10': 'tab:purple'}
fmice = ['F3', 'F4', 'F11', 'F9', 'F10']
mmice = ['M30', 'M1', 'M14', 'M3', 'M1bis']
sex = ['F', 'M']




In [10]:
mid = 'M'
exp_folder = '../../experiment/homeo/'

exp_data = load_object(exp_folder + 'data_ctr_' + mid + '.pkl')

livesize = exp_data['livesize']

clone_anal_M = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'S2like')

mid = 'F'
exp_folder = '../../experiment/homeo/'

exp_data = load_object(exp_folder + 'data_ctr_' + mid + '.pkl')

livesize = exp_data['livesize']

clone_anal_F = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'S2like')

In [14]:



castrated = ['M24_castre', 'M25_castre', 'M26_castre']

sex = ['castre', 'M', 'F', 'Ovx', 'Notch1_F', 'Notch1_M', 'Lgr6_F', 'Lgr6_M']


mdata = pd.DataFrame()

inv_wam_with_se = lambda v, se: np.array([np.average(1/v, weights=v**4/(se**2)), np.sqrt(np.sum(se**2/v**4))/len(v)])
wam_with_se = lambda v, se: np.array([np.average(v, weights=1/(se**2)), np.sqrt(np.sum(se**2))/len(v)])


for mid in mmice + fmice + castrated + sex:
    exp_folder = '../../experiment/homeo/'
    if 'castre' in mid:
        exp_folder = '../../experiment/castre/'
        exp_data = load_object(exp_folder + 'data_'     + mid + '.pkl')
    elif mid in ['Ovx']:
        exp_folder = '../../experiment/ovx/'
        exp_data = load_object(exp_folder + 'data_' + mid + '.pkl')
    elif mid in ['Notch1_F', 'Notch1_M']:
         exp_folder = '../../experiment/notch/'
         exp_data = load_object(exp_folder + 'data_' + mid + '.pkl')
    elif mid in ['Lgr6_F', 'Lgr6_M']:
         exp_folder = '../../experiment/lgr6/'
         exp_data = load_object(exp_folder + 'data_' + mid + '.pkl')         
    else:
        exp_data = load_object(exp_folder + 'data_ctr_' + mid + '.pkl')

    livesize = exp_data['livesize']

    clone_anal = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'S2like')


    Δλ, se_Δλ = bias_to_division(clone_anal)

    lambda_av = (clone_anal.av_tot_cs[0] - 1)/(7-2)
    lambda_se = clone_anal.se_tot_cs[0]/(7-2) + 0.5*lambda_av/5

    lcd1 = 1/clone_anal.lcd[0]
    sel1 = clone_anal.se_lcd[0]/clone_anal.lcd[0]**2

    ns2 = clone_anal.nclones

    se = clone_anal.se_lcd[3:-1]
    av = clone_anal.lcd[3:-1]
    fS = inv_wam_with_se(av, se)

    av = clone_anal.rel_sup_bas[3:-1]
    se = clone_anal.boot('rel_sup_bas', 500)[3:-1]
    nsb_b = wam_with_se(av, se)

    t0 = 3
    av = clone_anal.av_bas_cs[t0:]
    se = clone_anal.se_bas_cs[t0:]
    t = clone_anal.tvals[t0:]
    avb = wls_fit(t, av, se)

    t0 = 3
    av = clone_anal.av_bas_cs[:t0]
    se = clone_anal.se_bas_cs[:t0]
    t = clone_anal.tvals[:t0]
    bearly = wls_fit(t, av, se)



    clone_anal_D = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'Dlike')
    nd = clone_anal_D.nclones
    μ_fit, μ_err = get_mu(clone_anal_D)

    clone_anal_T = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'Tlike')
    nT = clone_anal_T.nclones

    clone_anal_T3 = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'T3like')
    nT3 = clone_anal_T3.nclones


    nclones = ns2 + nd

    ns3 = ns2 - nT
    ns4 = ns2 - nT - nT3


    clone_anal_S3p = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'Slike')

    se = clone_anal_S3p.se_lcd[3:-1]
    av = clone_anal_S3p.lcd[3:-1]
    fS3p = inv_wam_with_se(av, se)

    clone_anal_S4p = clones.xanalysis(livesize, verbose = False, tvals = exp_data['ctrTimeClonesize'], subpop = 'S4like')

    se = clone_anal_S4p.se_lcd[3:-1]
    av = clone_anal_S4p.lcd[3:-1]
    fS4p = inv_wam_with_se(av, se)  
    boot_lcd = clone_anal_S4p.boot('lcd', nboots = 1000, calc = None)
    avs = np.zeros(boot_lcd.shape[0])
    for i in range(1000):
        av = boot_lcd[i,3:-1]
        se = clone_anal_S4p.se_lcd[3:-1]
        fb = inv_wam_with_se(av, se)  
        avs[i]  = fb[0]
    fS4p[1] = np.std(avs)

    lambdaT3_av = (clone_anal_T3.av_tot_cs[0] - 1)/7
    lambdaT3_se = clone_anal_T3.se_tot_cs[0]/7

    lambda4_av = (clone_anal_S4p.av_tot_cs[0] - 1)/7
    lambda4_se = clone_anal_S4p.se_tot_cs[0]/7  

    lambdaT_av = (clone_anal_T.av_tot_cs[0] - 1)/7
    lambdaT_se = clone_anal_T.se_tot_cs[0]/7      

    mdata = pd.concat([mdata, pd.DataFrame([{
        'Mouse': mid,
        'Δ': (Δλ/lambda_av, se_Δλ/lambda_av),
        'fs': (fS[0], fS[1]),      
        'sb_rat': (nsb_b[0], nsb_b[1]),
        'λ': (lambda_av,   lambda_se),
        'fs3+': (fS3p[0], fS3p[1]),        
        'fs4+': (fS4p[0], fS4p[1]),    
        'lab2': (lcd1, sel1),     
        'rλ/f': (avb[0], avb[2]),
        'C0': (avb[1], avb[3]),
        'λT': (lambdaT_av, lambdaT_se),          
        'λT3': (lambdaT3_av, lambdaT3_se),          
        'λ4': (lambda4_av, lambda4_se),                
        'μ': (μ_fit, μ_err),
        'bearly': (bearly[0], bearly[2]),
        '#S2+': (ns2/nclones, np.sqrt((ns2/nclones)*(1 - ns2/nclones)/nclones)),
        '#S3+': (ns3/nclones, np.sqrt((ns3/nclones)*(1 - ns3/nclones)/nclones)),
        '#S4+': (ns4/nclones, np.sqrt((ns4/nclones)*(1 - ns4/nclones)/nclones)),
        '#D': (nd/nclones, np.sqrt((nd/nclones)*(1 - nd/nclones)/nclones)),
        '#T': (nT/nclones, np.sqrt((nT/nclones)*(1 - nT/nclones)/nclones)),
        '#T3': (nT3/nclones, np.sqrt((nT3/nclones)*(1 - nT3/nclones)/nclones)),
        'nclones': nclones
        }])], ignore_index=True)

# Calculate r:inf and its standard error
rinf = mdata['rλ/f'].apply(lambda x: x[0]) * mdata['fs'].apply(lambda x: x[0]) / mdata['λ'].apply(lambda x: x[0])
r_se = rinf * np.sqrt(
    (mdata['rλ/f'].apply(lambda x: x[1]/x[0]))**2 + 
    (mdata['fs'].apply(lambda x: x[1]/x[0]))**2 + 
    (mdata['λ'].apply(lambda x: x[1]/x[0]))**2
)
mdata['rinf'] = list(zip(rinf, r_se))

# Calculate μ:inf and its standard error
μinf = mdata['λ'].apply(lambda x: x[0]) * mdata['fs'].apply(lambda x: x[0]) / (1 - mdata['fs'].apply(lambda x: x[0]))
μinf_se = μinf * np.sqrt(
    (mdata['λ'].apply(lambda x: x[1]/x[0]))**2 + 
    (mdata['fs'].apply(lambda x: x[1]/x[0]))**2
)
mdata['μinf'] = list(zip(μinf, μinf_se))

μinf2 = mdata['λ'].apply(lambda x: x[0]) * (2*mdata['fs4+'].apply(lambda x: x[0])) / (1 - 2*mdata['fs4+'].apply(lambda x: x[0]))
μinf2_se = μinf * np.sqrt(
    (mdata['λ'].apply(lambda x: x[1]/x[0]))**2 + 
    (mdata['fs'].apply(lambda x: x[1]/x[0]))**2
)
mdata['μinf2'] = list(zip(μinf2, μinf2_se))

pd.set_option('display.max_columns', 500)
mdata

/home/joelh/Documents/Science/Projects/GlassyTissue/papers/StemTA_Homeostasis/squasim-homeostasis/notebooks/2_SD_model/../../analysis/clones.py:572: RuntimeWarning: invalid value encountered in divide
  self._av_tot_cs = np.sum(self.tot_ls*bas_mask*(self.tot_ls >= 0),0)/self.nsur
/tmp/ipykernel_214745/2721654887.py:104: RuntimeWarning: divide by zero encountered in log
  boot_Δλ[b], c, dm, dc = wls_fit(clone_anal.tvals[tlower:], np.log(lcd_boots[b,tlower:]), clone_anal.se_lcd[tlower:])
/home/joelh/miniforge3/envs/tisspy/lib/python3.12/site-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
/home/joelh/Documents/Science/Projects/GlassyTissue/papers/StemTA_Homeostasis/squasim-homeostasis/notebooks/2_SD_model/../../analysis/clones.py:1873: RuntimeWarning: divide by zero encountered in divide
  x = (self.av_full_tot_cs - self.lcd)/self.lcd
/home/joelh/Documents/Science/Projects/GlassyTissue/papers/StemTA_Homeostasis/s

,Mouse,Δ,fs,sb_rat,λ,fs3+,fs4+,lab2,rλ/f,C0,λT,λT3,λ4,μ,bearly,#S2+,#S3+,#S4+,#D,#T,#T3,nclones,rinf,μinf,μinf2
0,M30,"(0.07784141967239673, 0.0241907403412467)","(0.562432679156235, 0.136259706375182)","(0.36424033509132214, 0.0708132493206378)","(0.14166666666666666, 0.027287808074785543)","(0.34466430514966384, 0.07261616650386256)","(0.2437193012309437, 0.05290475538698937)","(0.6, 0.02449489742783177)","(0.09223202061451374, 0.0334861537122279)","(0.830022499721002, 1.4936654699318934)","(0.11111111111111109, 0.013998684185526933)","(0.1142857142857143, 0.018070158058105024)","(0.08571428571428573, 0.015649215928719037)","(nan, nan)","(0.08096433158944027, 0.0004933245389940274)","(0.8421052631578947, 0.04829804923590775)","(0.5263157894736842, 0.06613482762500456)","(0.3508771929824561, 0.06321258874556504)","(0.15789473684210525, 0.04829804923590773)","(0.3157894736842105, 0.06156817388058966)","(0.17543859649122806, 0.0503775039171214)",57,"(0.36617154676386393, 0.17469571805407266)","(0.18209303822511902, 0.056359607038922205)","(0.13472298606002497, 0.056359607038922205)"
1,M1,"(-0.04286064434783323, 0.05312034765726469)","(0.6835980370496565, 0.13577397264659333)","(0.3410740620866465, 0.05034627019278174)","(0.12653061224489798, 0.029672909060438974)","(0.47767888036261735, 0.09149367195391744)","(0.306622291580093, 0.089546892016488)","(0.6363636363636364, 0.035063146174539574)","(0.01104348324281542, 0.017326827027284726)","(5.622735973325989, 0.8459987149125234)","(0.08163265306122448, 0.01889428775046023)","(0.10714285714285714, 0.017857142857142856)","(0.08695652173913045, 0.02104321872943901)","(0.2602192554936134, 0.07528411588530234)","(0.1295477036693462, 0.03334137266202609)","(0.7313432835820896, 0.05415294608121959)","(0.5223880597014925, 0.06102345704883558)","(0.34328358208955223, 0.058006685713137024)","(0.26865671641791045, 0.05415294608121959)","(0.208955223880597, 0.049669495472589524)","(0.1791044776119403, 0.04684462467949339)",67,"(0.05966384998096619, 0.09538927105603337)","(0.27337402508744324, 0.08401274540062613)","(0.20062863811229698, 0.08401274540062613)"
2,M14,"(-0.05697303439564331, nan)","(0.8761096629053693, 0.14187035068217632)","(0.4956842179714519, 0.05190248095861581)","(0.13442622950819674, 0.025463666468061665)","(0.62259167977305, 0.0947372989247511)","(0.40896893271529866, 0.04891982532143268)","(0.61, 0.02287749986340289)","(0.013286423794631937, 0.011382564779039251)","(2.7086608464779327, 0.5657199587528173)","(0.10084033613445377, 0.015787167002146628)","(0.08928571428571429, 0.017290104224140256)","(0.09693877551020409, 0.012608486801753782)","(nan, nan)","(0.08436011332466684, 0.01950059382617882)","(0.7093023255813954, 0.04896516755606224)","(0.5116279069767442, 0.05390180475247302)","(0.32558139534883723, 0.05052953894914609)","(0.29069767441860467, 0.04896516755606225)","(0.19767441860465115, 0.042943884806284104)","(0.18604651162790697, 0.04196249808820081)",86,"(0.08659295373023226, 0.07725963470731217)","(0.9506158541655158, 0.23689949893378098)","(0.6039273541522927, 0.23689949893378098)"
3,M3,"(-0.02263490079919701, 0.020130073468063805)","(0.5175393433680036, 0.04649848419769502)","(0.37855042300726516, 0.03158349885345801)","(0.13999999999999999, 0.02837590576856521)","(0.3610402931939561, 0.028279277191040612)","(0.32713622922170926, 0.03574659649213963)","(0.5882352941176471, 0.02487180928817511)","(0.02695680547309157, 0.007908793461862294)","(2.365719495388289, 0.4031333950661771)","(0.10084033613445377, 0.015787167002146628)","(0.14285714285714285, 0.0)","(0.09523809523809525, 0.014028236011081873)","(nan, nan)","(0.11083444346804959, 0.005313311336441095)","(0.7894736842105263, 0.046764385086244135)","(0.5657894736842105, 0.05685528086757625)","(0.5131578947368421, 0.057334070632943414)","(0.21052631578947367, 0.046764385086244135)","(0.2236842105263158, 0.04780024010173328)","(0.05263157894736842, 0.025613908599594084)",76,"(0.09965148145602012, 0.03664

In [ ]:
mdata[['Mouse', 'nclones', '#S4+', '#T3', '#T', '#D'] ]

,Mouse,nclones,#S4+,#T3,#T,#D
0,M30,57,"(0.3508771929824561, 0.06321258874556504)","(0.17543859649122806, 0.0503775039171214)","(0.3157894736842105, 0.06156817388058966)","(0.15789473684210525, 0.04829804923590773)"
1,M1,67,"(0.34328358208955223, 0.058006685713137024)","(0.1791044776119403, 0.04684462467949339)","(0.208955223880597, 0.049669495472589524)","(0.26865671641791045, 0.05415294608121959)"
2,M14,86,"(0.32558139534883723, 0.05052953894914609)","(0.18604651162790697, 0.04196249808820081)","(0.19767441860465115, 0.042943884806284104)","(0.29069767441860467, 0.04896516755606225)"
3,M3,76,"(0.5131578947368421, 0.057334070632943414)","(0.05263157894736842, 0.025613908599594084)","(0.2236842105263158, 0.04780024010173328)","(0.21052631578947367, 0.046764385086244135)"
4,M1bis,98,"(0.47959183673469385, 0.050465537530943905)","(0.02040816326530612, 0.0142827390249748)","(0.21428571428571427, 0.041449175533856936)","(0.2857142857142857, 0.04563404035713856)"
5,F3,149,"(0.7315436241610739, 0.036304782729866154)","(0.040268456375838924, 0.016105130242060513)","(0.10738255033557047, 0.02536333799446498)","(0.12080536912751678, 0.026698846456930426)"
6,F4,239,"(0.4811715481171548, 0.03231937184803326)","(0.058577405857740586, 0.015190022790896067)","(0.13389121338912133, 0.022027403659826306)","(0.3263598326359833, 0.030329369907510364)"
7,F11,109,"(0.44036697247706424, 0.04754948190694402)","(0.0, 0.0)","(0.28440366972477066, 0.04321041141655161)","(0.27522935779816515, 0.04277937409248436)"
8,F9,119,"(0.40336134453781514, 0.04497066927131235)","(0.03361344537815126, 0.016521842346441974)","(0.15966386554621848, 0.03357811857345877)","(0.40336134453781514, 0.04497066927131235)"
9,F10,129,"(0.4108527131782946, 0.043317178973290696)","(0.09302325581395349, 0.02557401929158873)","(0.24806201550387597, 0.038025618859851014)","(0.24806201550387597, 0.038025618859851014)"
